# MinerU Reader Demo

This notebook demonstrates how to use `MinerUReader` to parse documents via the [MinerU](https://mineru.net) API.

MinerU provides two parsing modes:
- **Flash** (default): No token needed, fast, Markdown-only output
- **Precision**: Token required, supports OCR, formula/table recognition, richer output

## Install Dependencies

In [ ]:
!pip install -qU llama-index-readers-mineru llama-index-core

'pip' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


## 1. Flash Mode (No Token Required)

The simplest way to get started — no API token needed.

In [ ]:
from llama_index.readers.mineru import MinerUReader

reader = MinerUReader()
print(f"Mode: {reader.mode}")

In [ ]:
# Parse a PDF from a public URL
documents = reader.load_data(
    "https://cdn-mineru.openxlab.org.cn/demo/example.pdf"
)

print(f"Number of documents: {len(documents)}")
print(f"Metadata: {documents[0].metadata}")
print(f"Content preview:\n{documents[0].text[:500]}")

## 2. Flash Mode with Per-Page Splitting

Set `split_pages=True` to get one Document per PDF page — ideal for RAG pipelines.

In [ ]:
reader_split = MinerUReader(split_pages=True, pages="1-3")

documents = reader_split.load_data(
    "https://cdn-mineru.openxlab.org.cn/demo/example.pdf"
)

print(f"Number of documents (pages): {len(documents)}")
for doc in documents:
    print(f"  Page {doc.metadata['page']}: {doc.text[:80]}...")

## 3. Precision Mode (Token Required)

For higher accuracy, OCR, formula/table recognition, and larger files.

Get your free token from https://mineru.net/apiManage/token

In [ ]:
import os

# Option 1: Set token via environment variable
# os.environ["MINERU_TOKEN"] = "your-api-token"

# Option 2: Pass token directly
TOKEN = "your-api-token"  # Replace with your actual token

precision_reader = MinerUReader(
    mode="precision",
    token=TOKEN,
    ocr=True,
    formula=True,
    table=True,
    language="en",
)

documents = precision_reader.load_data(
    "https://cdn-mineru.openxlab.org.cn/demo/example.pdf"
)

print(f"Number of documents: {len(documents)}")
print(f"Metadata: {documents[0].metadata}")
print(f"Content preview:\n{documents[0].text[:500]}")

## 4. Multiple Files at Once

In [ ]:
reader = MinerUReader()

documents = reader.load_data([
    "https://cdn-mineru.openxlab.org.cn/demo/example.pdf",
    # "/path/to/local_file.pdf",
])

for doc in documents:
    print(f"Source: {doc.metadata['source']}")
    print(f"Content length: {len(doc.text)} chars\n")

## 5. Build a RAG Pipeline with VectorStoreIndex

Combine MinerU parsing with LlamaIndex's indexing and querying.

In [ ]:
# Uncomment and run if you have an OpenAI API key set up
# from llama_index.core import VectorStoreIndex
#
# reader = MinerUReader()
# documents = reader.load_data("https://cdn-mineru.openxlab.org.cn/demo/example.pdf")
#
# index = VectorStoreIndex.from_documents(documents)
# query_engine = index.as_query_engine()
# response = query_engine.query("Summarize the key findings")
# print(response)